# table of content  

In [ ]:

import ansys.aedt.core
import os
import tempfile
import time

AEDT_VERSION = "2025.1"
NUM_CORES = 8
NG_MODE = True  # Open AEDT UI when it is launched.
# project파일의 경로는
pjtPath=r"E:\KDH\WPT\wpt8-53.aedt"
# 위에꺼 반영해줘

from ansys.aedt.core import Maxwell3d

m3d=Maxwell3d(project=pjtPath, design="GA", solution_type="EddyCurrent", specified_version=AEDT_VERSION, new_desktop_session=False)


In [ ]:
m3d.set_active_design("GA" )

In [ ]:
varManager=m3d.variable_manager

In [ ]:
try:
    design_vars = m3d.variable_manager.design_variables
except Exception as e:
    print("디자인 변수 조회 중 오류:", e)
    design_vars = {}

# 1) 파라미터 정의 (슬라이드 10–11, 27–28) ----------


In [ ]:

# GA-side
GA = dict(
    GA_Coil_Wi_x="140mm", GA_Coil_Wi_y="290mm",
    GA_Coil_Wo_x="500mm", GA_Coil_Wo_y="650mm",
    GA_Coil_Turn="8", GA_Cond_W="10mm", GA_Cond_T="5mm",
    GA_Cond_G_x="(GA_Coil_Wo_x/2-GA_Coil_Wi_x/2-GA_Cond_W*GA_Coil_Turn)/(GA_Coil_Turn-1)",
    GA_Cond_G_y="(GA_Coil_Wo_y/2-GA_Coil_Wi_y/2-GA_Cond_W*GA_Coil_Turn)/(GA_Coil_Turn-1)",
    GA_Ferrite_W_x="500mm", GA_Ferrite_W_y="650mm", GA_Ferrite_T="5mm",
    GA_Aluminum_W_x="750mm", GA_Aluminum_W_y="750mm", GA_Aluminum_T="3mm",
)
# VA-side
VA = dict(
    VA_Coil_Wi_x="130mm", VA_Coil_Wi_y="130mm",
    VA_Coil_Wo_x="270mm", VA_Coil_Wo_y="270mm",
    VA_Coil_Turn="10", VA_Cond_W="5mm", VA_Cond_T="5mm",
    VA_Cond_G_x="(VA_Coil_Wo_x/2-VA_Coil_Wi_x/2-VA_Cond_W*VA_Coil_Turn)/(VA_Coil_Turn-1)",
    VA_Cond_G_y="(VA_Coil_Wo_y/2-VA_Coil_Wi_y/2-VA_Cond_W*VA_Coil_Turn)/(VA_Coil_Turn-1)",
    VA_Ferrite_W_x="300mm", VA_Ferrite_W_y="300mm", VA_Ferrite_T="5mm",
    VA_Aluminum_W_x="750mm", VA_Aluminum_W_y="750mm", VA_Aluminum_T="2mm",
)
# 공통(오프셋/갭/영역)
common = dict(
    Z_gap="130mm",           # Z1-class의 예시 지상고(교육자료 용어 참조)
    Offset_X="0mm", Offset_Y="0mm",  # 자연/지정 오프셋용 변수 (슬라이드 41–44 컨셉 반영)
    Air_Pad="150mm"          # 시뮬레이션 영역 여유치 (슬라이드 45 맥락)
)
# design properties로 정의


In [ ]:
for key, value in GA.items():
    varManager.set_variable(key, value, "Design")
for key, value in VA.items():
    varManager.set_variable(key, value, "Design")
for key, value in common.items():
    varManager.set_variable(key, value, "Design")

In [ ]:
prefix="GA"
coil_name = f"{prefix}_Coil"


In [ ]:
## 밑에 함수를 실행하면 design properties의 variables로부터 데이터를 가져오도록 바꿔줄래?   
#value를 불러오도록 바꿔야될껄  
outer_box_obj = m3d.modeler.create_box(
    origin=[f"-{design_vars['GA_Coil_Wo_x'].value}/2", f"-{design_vars['GA_Coil_Wo_y'].value}/2", "0mm"], 
    sizes=(
        f"{design_vars['GA_Coil_Wo_x'].value}",
        f"{design_vars['GA_Coil_Wo_y'].value}",
        f"{design_vars['GA_Cond_T'].value}"),
        name=coil_name
)


In [ ]:
GA

In [ ]:
# ---------- 3) 유틸: 복잡한 직사각 프레임 코일 생성기 ----------
# 위에 셀을 참고해서 문법이 바뀐걸 참고해서 함수를
# 함수내의 변수명들을 셀 5의 변수들을 고려해서 수정해줘
#def build_complex_frame_coil(prefix, copper_name="copper"):
prefix="GA"   
copper_name="copper"
coil_name = f"{prefix}_Coil"
# 셀 5에서 정의된 변수명들을 직접 문자열로 사용 (AEDT 파라미터로 인식)
Wi_x= design_vars['GA_Coil_Wi_x'].value*1000
Wi_y = design_vars['GA_Coil_Wi_y'].value*1000
Wo_x = design_vars['GA_Coil_Wo_x'].value*1000
Wo_y = design_vars['GA_Coil_Wo_y'].value*1000
turn = design_vars['GA_Coil_Turn'].value
Cond_W = design_vars['GA_Cond_W'].value*1000 # Width
Cond_T = design_vars['GA_Cond_T'].value*1000  # Thickness
Cond_G_x = design_vars['GA_Cond_G_x'].value*1000
Cond_G_y = design_vars['GA_Cond_G_y'].value*1000


bottom_start_y=design_vars['GA_Coil_Wi_y']/2

# print(f"Creating {coil_name} with turn-based duplication...")

#bottom bars
bar_bottom = m3d.modeler.create_box(  # 내부 하단에서 시작
    origin=['GA_Coil_Wo_x/2', 'GA_Coil_Wo_y/2', "0mm"],
    sizes=["GA_Coil_Wo_x", "GA_Cond_W", "GA_Cond_T"],
    name=f"{prefix}_bar_bottom",material=copper_name
    )

m3d.modeler.duplicate_along_line(
        assignment='GA_bar_bottom',
        vector=[0,"GA_Cond_G_y+GA_Cond_W","0mm"],
        clones=int(turn)+1,attach=True)

# right bars
bar_top = m3d.modeler.create_box(  # 내부 상단에서 시작
    origin=['GA_Coil_Wo_x/2', 'GA_Coil_Wo_y/2', "0mm"],
    sizes=["GA_Cond_W", "GA_Coil_Wo_y", "GA_Cond_T"],
    name=f"{prefix}_bar_right",material=copper_name
    )

m3d.modeler.duplicate_along_line(
        assignment=f"{prefix}_bar_right",
        vector=["GA_Cond_G_x+GA_Cond_W","0mm","0mm"],
        clones=int(turn),attach=True)

# top bars
bar_top = m3d.modeler.create_box(  # 내부 상단에서 시작
    origin=['-GA_Coil_Wo_x/2', 'GA_Coil_Wo_y/2', "0mm"],
    sizes=["GA_Cond_W", "-GA_Cond_W", "GA_Cond_T"],
    name=f"{prefix}_bar_top",material=copper_name
    )

m3d.modeler.duplicate_along_line(
        assignment=f"{prefix}_bar_top",
        vector=[0,"-(GA_Cond_G_y+GA_Cond_W)","0mm"],
        clones=int(turn),attach=True)


# left bars
bar_left = m3d.modeler.create_box(  # 내부 상단에서 시작
    origin=['GA_Coil_Wo_x/2', '-GA_Coil_Wo_y/2', "0mm"],
    sizes=["-GA_Cond_W", "GA_Coil_Wo_y", "GA_Cond_T"],
    name=f"{prefix}_bar_left",material=copper_name
    )

m3d.modeler.duplicate_along_line(
        assignment=f"{prefix}_bar_left",
        vector=["-GA_Cond_G_x-GA_Cond_W","0mm","0mm"],
        clones=int(turn),attach=True)


# Side Coil Draw


In [ ]:
# Draw Line과 Sweep Along Vector 구현
# Manual Drawing Process를 PyAEDT로 자동화

# 1) 4개의 점을 연결하는 폴리라인 생성
# Point 좌표들 (design properties variables 사용)
points = [
    # 1st Line: Point1 -> Point2
    ["-GA_Coil_Wo_x/2", "-GA_Coil_Wo_y/2+GA_Cond_G_y+GA_Cond_W", "0mm"],
    ["-GA_Coil_Wi_x/2", "-GA_Coil_Wi_y/2+GA_Cond_G_y+GA_Cond_W", "0mm"],
    
    # 2nd Line: 이어서
    
    ["-GA_Coil_Wi_x/2", "GA_Coil_Wi_y/2", "0mm"],
    
    # 3rd Line: 이어서  
    ["-GA_Coil_Wo_x/2", "GA_Coil_Wo_y/2", "0mm"],
    
    # 4th Line: 시작점으로 돌아가서 닫기
    ["-GA_Coil_Wo_x/2", "-GA_Coil_Wo_y/2+GA_Cond_G_y+GA_Cond_W", "0mm"]
]


In [ ]:

print("Creating polyline with design properties variables...")

# 2) 폴리라인 생성 (Sheets -> Unassigned에 생성됨)
polyline1 = m3d.modeler.create_polyline(
    points=points,
    cover_surface=True,
    close_surface=True,
    name="Polyline1"
)


In [ ]:

print(f"Created polyline: {polyline1}")

# 3) Sweep Along Vector 실행
# Vector: 0mm, 0mm, GA_Cond_T (Z방향으로 두께만큼)
sweep_vector = ["0mm", "0mm", "GA_Cond_T"]

print("Performing Sweep Along Vector...")

try:
    # Sweep 작업 실행
    swept_object = m3d.modeler.sweep_along_vector(
        assignment=polyline1,
        vector=sweep_vector,
        draft_angle=0.0,
        name="SweptCoilSection"
    )
    
    print(f"Successfully created swept object: {swept_object}")
    print(f"Vector used: {sweep_vector}")
    
except Exception as e:
    print(f"Error during sweep operation: {e}")
    
print("Draw Line and Sweep Along Vector process completed!")

In [ ]:
# 두 번째 폴리라인 생성 - 상단 라인들
# 1st CreateLine: Point1 -> Point2
# 2nd CreateLine: 이어서
# 3rd CreateLine: 이어서  
# 4th CreateLine: 시작점으로 돌아가서 닫기

points2 = [
    # 1st CreateLine: Point1 -> Point2
    ["-GA_Coil_Wo_x/2", "GA_Coil_Wo_y/2", "0mm"],
    ["-GA_Coil_Wi_x/2", "GA_Coil_Wi_y/2", "0mm"],
    
    # 2nd CreateLine: 이어서
    ["GA_Coil_Wi_x/2", "GA_Coil_Wi_y/2", "0mm"],
    
    # 3rd CreateLine: 이어서
    ["GA_Coil_Wo_x/2", "GA_Coil_Wo_y/2", "0mm"],
    
    # 4th CreateLine: 시작점으로 돌아가서 닫기
    ["-GA_Coil_Wo_x/2", "GA_Coil_Wo_y/2", "0mm"]
]

print("Creating second polyline with design properties variables...")

# 두 번째 폴리라인 생성 (Sheets -> Unassigned에 생성됨)
polyline2 = m3d.modeler.create_polyline(
    points=points2,
    cover_surface=True,
    close_surface=True,
    name="Polyline2"
)

print(f"Created second polyline: {polyline2}")

# Sweep Along Vector 실행
print("Performing Sweep Along Vector for second polyline...")


In [ ]:
m3d.modeler.swee

In [ ]:

try:
    # Sweep 작업 실행
    swept_object2 = m3d.modeler.sweep_along_vector(
        assignment="Polyline2",
        sweep_vector=["0mm", "0mm", "GA_Cond_T"],
    )
    
    print(f"Successfully created second swept object: {swept_object2}")
    
except Exception as e:
    print(f"Error during second sweep operation: {e}")
    
print("Second Draw Line and Sweep Along Vector process completed!")

In [136]:
bars_va = []
# 하단 가로
for i in range(int(m3d["VA_Coil_Turn"].value) + 1):
    y0 = "-VA_Coil_Wo_y/2 + {}*({}+{})".format(i, "VA_Cond_W", "VA_Cond_G_y")
    bars_va.append(
        m3d.modeler.create_box(
            position=["-VA_Coil_Wo_x/2", y0, "0mm"],
            dimensions_list=["VA_Coil_Wo_x", "VA_Cond_W", "VA_Cond_T"],
            name=f"VA_Bot_{i}",
            material="copper",
        )
    )

AttributeError: 'str' object has no attribute 'value'

In [ ]:
        # 2-2) 모든 바들을 Unite하여 단일 객체로 만들기
        print("Uniting all bars...")
        if len(all_objects) > 1:
            coil_union = m3d.modeler.unite(all_objects, keep_originals=False)
        else:
            coil_union = all_objects[0]
            
        # 2-3) 머티리얼 & 이름 할당
        try:
            if hasattr(coil_union, 'name'):
                coil_union.name = coil_name
                coil_union.material_name = copper_name
            else:
                # 객체 참조 방식으로 할당
                m3d.modeler[coil_union].name = coil_name
                m3d.modeler[coil_union].material_name = copper_name
        except Exception as e:
            print(f"Warning: Material assignment failed: {e}")
            
        print(f"Successfully created complex coil: {coil_name}")
        return coil_union
        
    except Exception as e:
        print(f"Error in complex coil creation: {e}")
        print("Falling back to simple frame coil...")
        
        # 실패 시 간단한 프레임 코일로 대체
        return build_simple_frame_coil(prefix, copper_name)

    # 기본 반환값
    return coil_name

In [ ]:
GA_Coil=build_complex_frame_coil(
    prefix="GA", copper_name="copper" )

In [ ]:
    def sweep_edge(poly_name, pts):
        try:
            pl = m3d.modeler.create_polyline(pts, close_polygon=False, cover_surface=False, name=poly_name)
            # 시트를 Z로 두께만큼 Extrude
            return m3d.modeler.sweep_along_vector(pl, [0, 0, t], draft_angle=0.0)
        except Exception as e:
            print(f"Warning: Failed to create edge sweep {poly_name}: {e}")
            return None

    edge_tools = []
    # 좌하 모서리 → 내곽 하변 → 내곽 좌변 → 외곽 좌상 모서리 → 외곽 상변 → 좌상귀환
    e1 = sweep_edge(f"{prefix}_edge1", [
        [f"-{Wo_x}/2", f"-{Wo_y}/2+{gy}+{w}", "0mm"],
        [f"-{Wi_x}/2", f"-{Wi_y}/2+{gy}+{w}", "0mm"],
        [f"-{Wi_x}/2", f"{Wi_y}/2", "0mm"],
        [f"-{Wo_x}/2", f"{Wo_y}/2", "0mm"],
        [f"-{Wo_x}/2", f"-{Wo_y}/2+{gy}+{w}", "0mm"],
    ])
    if e1: edge_tools.append(e1)

    e2 = sweep_edge(f"{prefix}_edge2", [
        [f"-{Wo_x}/2", f"{Wo_y}/2", "0mm"],
        [f"-{Wi_x}/2", f"{Wi_y}/2", "0mm"],
        [f"{Wi_x}/2",  f"{Wi_y}/2", "0mm"],
        [f"{Wo_x}/2",  f"{Wo_y}/2", "0mm"],
        [f"-{Wo_x}/2", f"{Wo_y}/2", "0mm"],
    ])
    if e2: edge_tools.append(e2)

    e3 = sweep_edge(f"{prefix}_edge3", [
        [f"{Wo_x}/2", f"{Wo_y}/2", "0mm"],
        [f"{Wi_x}/2", f"{Wi_y}/2", "0mm"],
        [f"{Wi_x}/2", f"-{Wi_y}/2", "0mm"],
        [f"{Wo_x}/2", f"-{Wo_y}/2", "0mm"],
        [f"{Wo_x}/2", f"{Wo_y}/2", "0mm"],
    ])
    if e3: edge_tools.append(e3)

    # 2-3) 네 방향 박스와 엣지 슬리브를 교차/삭감하여 프레임 만들기
    try:
        # 모든 솔리드 객체를 Unite
        all_bars = [obj for obj in m3d.modeler.get_objects_in_group("Solids") 
                   if obj.startswith(prefix)]
        if all_bars:
            coil_union = m3d.modeler.unite(all_bars, keep_originals=False)
            
            # 내측 빈 영역 커팅용 시트들로 Subtract
            for tool in edge_tools:
                try:
                    m3d.modeler.subtract(coil_union, tool, keep_originals=False)
                except Exception as e:
                    print(f"Warning: Subtract operation failed: {e}")
                    
            # 머티리얼 & 이름 할당
            if hasattr(coil_union, 'name'):
                coil_union.name = coil_name
                coil_union.material_name = copper_name
            else:
                # 객체 참조 방식으로 할당
                m3d.modeler[coil_union].name = coil_name
                m3d.modeler[coil_union].material_name = copper_name
            
            print(f"Successfully created complex coil: {coil_name}")
            return coil_union
            
    except Exception as e:
        print(f"Error in complex coil creation: {e}")
        print("Falling back to simple frame coil...")
        
        # 실패 시 간단한 프레임 코일로 대체
        return build_simple_frame_coil(prefix, copper_name)

    # 기본 반환값


In [ ]:
# ---------- 복잡한 코일 생성 함수의 확장 부분 (필요시 사용) ----------
def create_advanced_spiral_coil(prefix, copper_name="copper"):
    """
    더 정교한 나선형 코일을 생성하는 함수 (향후 확장용)
    현재는 build_complex_frame_coil이 메인 함수로 사용됨
    """
    coil_name = f"{prefix}_Advanced_Coil"
    
    # 여기에 향후 더 복잡한 나선형 코일 로직을 구현할 수 있음
    # 현재는 간단한 프레임 코일로 대체
    return build_simple_frame_coil(prefix, copper_name)

In [ ]:
# ---------- 3) GA 코일/포트/자성체/실드판 (슬라이드 12–26, 23–25) ----------
GA_Coil = build_complex_frame_coil(
    prefix="GA", copper_name="copper"
)

# 포트 시트 + 전류 여기 (슬라이드 23)
GA_Port = m3d.modeler.create_rectangle(cs_plane="XY", position=[f"-GA_Cond_T/2", f"-GA_Coil_Wo_y/2", f"GA_Cond_T+2.5mm"],
                                        dimension_list=[f"GA_Cond_T", f"GA_Cond_W"], name="GA_Port")
m3d.assign_current(GA_Port, amplitude=1.0, phase=0.0, solid=False, name="Current_GA")

# FDKH40 자성체 추가 + 물성 (슬라이드 24)
if "FDKH40" not in m3d.materials.material_keys:
    m3d.materials.add_material("FDKH40")
    m3d.materials["FDKH40"].permittivity = 1
    m3d.materials["FDKH40"].permeability = 2400
    m3d.materials["FDKH40"].conductivity = 0
    m3d.materials["FDKH40"].magnetic_loss_tangent = 0.0024

GA_Ferrite = m3d.modeler.create_box(position=[f"-GA_Ferrite_W_x/2", f"-GA_Ferrite_W_y/2", "0mm"],
                                    xsize="GA_Ferrite_W_x", ysize="GA_Ferrite_W_y", zsize="GA_Ferrite_T", name="GA_Ferrite")
m3d.modeler[GA_Ferrite].material_name = "FDKH40"

# 알루미늄 실드판 (슬라이드 25)
GA_Al = m3d.modeler.create_box(position=[f"-GA_Aluminum_W_x/2", f"-GA_Aluminum_W_y/2", "0mm"],
                               xsize="GA_Aluminum_W_x", ysize="GA_Aluminum_W_y", zsize="GA_Aluminum_T", name="GA_Aluminum")
m3d.modeler[GA_Al].material_name = "aluminum"

# GA 이동 (슬라이드 26)
m3d.modeler.move([GA_Coil, GA_Port], [0, 0, "GA_Aluminum_T+GA_Ferrite_T+2mm"])
m3d.modeler.move(GA_Ferrite, [0, 0, "GA_Aluminum_T+1mm"])

# ---------- 4) VA 파라미터 & 코일/포트/자성체/실드판 (슬라이드 27–38) ----------
VA_Coil = build_complex_frame_coil(
    prefix="VA", copper_name="copper"
)

VA_Port = m3d.modeler.create_rectangle(cs_plane="XY",
                                        position=[f"-VA_Cond_T/2", f"-VA_Coil_Wo_y/2", f"VA_Cond_T+2.5mm"],
                                        dimension_list=[f"VA_Cond_T", f"VA_Cond_W"], name="VA_Port")
m3d.assign_current(VA_Port, amplitude=1.0, phase=0.0, solid=False, name="Current_VA")

VA_Ferrite = m3d.modeler.create_box(position=[f"-VA_Ferrite_W_x/2", f"-VA_Ferrite_W_y/2", "0mm"],
                                    xsize="VA_Ferrite_W_x", ysize="VA_Ferrite_W_y", zsize="VA_Ferrite_T", name="VA_Ferrite")
m3d.modeler[VA_Ferrite].material_name = "FDKH40"

VA_Al = m3d.modeler.create_box(position=[f"-VA_Aluminum_W_x/2", f"-VA_Aluminum_W_y/2", "0mm"],
                               xsize="VA_Aluminum_W_x", ysize="VA_Aluminum_W_y", zsize="VA_Aluminum_T", name="VA_Aluminum")
m3d.modeler[VA_Al].material_name = "aluminum"

# ---------- 5) 오프셋/배치 & 시뮬레이션 영역 (슬라이드 41–45) ----------
# GA는 z=0 근처(실드/페라이트/코일 위로 배치됨)
# VA는 GA 위 Z_gap 만큼 위쪽에 배치 + XY 오프셋
m3d.modeler.move([VA_Coil, VA_Port, VA_Ferrite, VA_Al],
                 ["Offset_X", "Offset_Y", f"GA_Aluminum_T+GA_Ferrite_T+Z_gap+10mm"])

# 공기영역/리전 (AirBox) 생성: 모델 전체 + 여유 Air_Pad mm
bb = m3d.modeler.get_model_bounding_box()
pad = "Air_Pad"  # 변수명을 문자열로 사용
region = m3d.modeler.create_region(pad_value=[pad, pad, pad, pad, pad, pad], is_percentage=False, name="Region")

# ---------- 6) Matrix & Eddy Effects (슬라이드 46–47) ----------
# 행렬: 두 전류여기(포트 시트)에 대한 2x2 임피던스/인덕턴스 행렬
try:
    mtx = m3d.assign_matrix([GA_Coil, VA_Coil], matrix_name="Matrix1")
except Exception:
    # 대안: 이름 기반 또는 Excitation 기반
    try:
        mtx = m3d.matrices.add_conductor_matrix("Matrix1", conductor_names=[GA_Coil, VA_Coil])
    except Exception as e:
        print(f"Warning: Failed to assign matrix: {e}")

# Eddy Effects: 도체 및 실드에 대하여 유/무 유효화
def set_eddy(obj_list, on=True):
    for obj in obj_list:
        try:
            m3d.assign_eddy_effect(obj, enabled=on, consider_displacement_current=on)
        except Exception:
            pass

set_eddy([GA_Coil, VA_Coil, GA_Al, VA_Al], on=True)

# ---------- 7) 메시 & 솔루션 셋업 (슬라이드 48–51, 49) ----------
# 표피두께 기반 메시조작: 구리/알루미늄에 국부 사이즈 제한
for obj in [GA_Coil, VA_Coil, GA_Al, VA_Al]:
    try:
        m3d.mesh.assign_length_mesh(obj, maximum_length="5mm", name=f"lm_{obj}")
    except Exception:
        pass

# Setup1 생성/수정 (슬라이드 49)
if "Setup1" not in m3d.analysis_setups:
    setup = m3d.create_setup("Setup1")
else:
    setup = m3d.get_setup("Setup1")
setup.props["MaximumPasses"] = 20
setup.props["Frequency"] = "85kHz"  # J2954 실습 주파수 맥락
setup.props["EnableSolverDomains"] = False
setup.props["SmoothBHCurve"] = True
setup.update()

# ---------- 8) 유효성 체크, 메시 생성, 해석, 결과 (슬라이드 50–53, 52–53) ----------
m3d.validate_full_design()         # Validation Check
m3d.analyze_nominal(num_cores=4)   # Generate Mesh + Solve (필요 시 코어수 조정)

# (결과 예시) Matrix → L, M, Z 확인 / 옴손실 적분 (슬라이드 52–53의 결과 분석 맥락)
try:
    # 등가회로 파라미터
    matrices = m3d.post.get_solution_data(expressions="Matrix1.L(GA_Coil,GA_Coil)")
    L1 = matrices.data_real()
    matrices = m3d.post.get_solution_data(expressions="Matrix1.L(VA_Coil,VA_Coil)")
    L2 = matrices.data_real()
    matrices = m3d.post.get_solution_data(expressions="Matrix1.M(GA_Coil,VA_Coil)")
    M12 = matrices.data_real()

    print("Self L(GA) [H]  :", L1[-1] if L1 else None)
    print("Self L(VA) [H]  :", L2[-1] if L2 else None)
    print("Mutual M [H]    :", M12[-1] if M12 else None)
except Exception:
    pass

# 옴손실 총합(코일/실드)
try:
    loss_ga = m3d.post.get_scalar_field_value("OhmicLoss", geometry=GA_Coil, report_type="Volume")
    loss_va = m3d.post.get_scalar_field_value("OhmicLoss", geometry=VA_Coil, report_type="Volume")
    loss_al = m3d.post.get_scalar_field_value("OhmicLoss", geometry=GA_Al,  report_type="Volume")
    print("Ohmic Loss [W] GA/VA/GA_Al :", loss_ga, loss_va, loss_al)
except Exception:
    pass

m3d.save_project()
print("Done: p.8–53 workflow scripted.")

In [137]:
m3d.close_desktop( )

PyAEDT INFO: Desktop has been released and closed.


True